# Member D: College production

**Tasks**

- [X] Get a CFBD key, put it in `.env`, run `python -m src.data.pull_college --check` for coverage by season.
- [X] Full pull, then zip `data/raw/college/` and upload to the team Drive.
- [X] Per position group: available stats, coverage, missingness by year and school level.
- [X] Write `reports/college_feature_audit.md`. Flag OL and other positions with almost no stats.

See `docs/week1_tasks.md` and the project plan for details. Reusable code goes in `src/`.

In [1]:
import sys; sys.path.append('..')
import pandas as pd
from src.utils import *

In [2]:
# this just gives me an error, so I'm guessing it will be built later
# master = pd.read_parquet(PROCESSED / 'master.parquet')
# master.head()

In [3]:
stats = pd.concat([pd.read_parquet(f) for f in sorted(RAW_COLLEGE.glob("player_stats_*.parquet"))], ignore_index=True)
picks = pd.concat([pd.read_parquet(f) for f in sorted(RAW_COLLEGE.glob("draft_picks_*.parquet"))], ignore_index=True)

stats.info()
stats.head()

<class 'pandas.DataFrame'>
RangeIndex: 871864 entries, 0 to 871863
Data columns (total 9 columns):
 #   Column      Non-Null Count   Dtype
---  ------      --------------   -----
 0   season      871864 non-null  int64
 1   playerId    871864 non-null  str  
 2   player      871864 non-null  str  
 3   position    871864 non-null  str  
 4   team        871864 non-null  str  
 5   conference  871864 non-null  str  
 6   category    871864 non-null  str  
 7   statType    871864 non-null  str  
 8   stat        871864 non-null  str  
dtypes: int64(1), str(8)
memory usage: 104.4 MB


,season,playerId,player,position,team,conference,category,statType,stat
0,2007,123339,Ryan Donahue,P,Iowa,Big Ten,punting,In 20,0
1,2007,123339,Ryan Donahue,P,Iowa,Big Ten,punting,LONG,82
2,2007,123339,Ryan Donahue,P,Iowa,Big Ten,punting,NO,86
3,2007,123339,Ryan Donahue,P,Iowa,Big Ten,punting,TB,0
4,2007,123339,Ryan Donahue,P,Iowa,Big Ten,punting,YDS,3533


In [4]:
picks.info()
picks.head()

<class 'pandas.DataFrame'>
RangeIndex: 3821 entries, 0 to 3820
Data columns (total 19 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   collegeAthleteId         3737 non-null   float64
 1   nflAthleteId             3821 non-null   int64  
 2   collegeId                3821 non-null   int64  
 3   collegeTeam              3821 non-null   str    
 4   collegeConference        3763 non-null   str    
 5   nflTeamId                3821 non-null   int64  
 6   nflTeam                  3821 non-null   str    
 7   year                     3821 non-null   int64  
 8   overall                  3821 non-null   int64  
 9   round                    3821 non-null   int64  
 10  pick                     3821 non-null   int64  
 11  name                     3821 non-null   str    
 12  position                 3821 non-null   str    
 13  height                   3771 non-null   float64
 14  weight                   3794 non-n

,collegeAthleteId,nflAthleteId,collegeId,collegeTeam,collegeConference,nflTeamId,nflTeam,year,overall,round,pick,name,position,height,weight,preDraftRanking,preDraftPositionRanking,preDraftGrade,hometownInfo
0,2131833.0,11405,99,LSU,SEC,13,Las Vegas,2007,1,1,1,JaMarcus Russell,Quarterback,78.0,256.0,2.0,1.0,98.0,"{'city': None, 'state': None, 'country': None,..."
1,2165223.0,11326,59,Georgia Tech,ACC,8,Detroit,2007,2,1,2,Calvin Johnson,Wide Receiver,77.0,239.0,1.0,1.0,99.0,"{'city': None, 'state': None, 'country': None,..."
2,2126512.0,11207,275,Wisconsin,Big Ten,5,Cleveland,2007,3,1,3,Joe Thomas,Offensive Tackle,79.0,311.0,4.0,1.0,98.0,"{'city': None, 'state': None, 'country': None,..."
3,2080614.0,10433,228,Clemson,ACC,27,Tampa Bay,2007,4,1,4,Gaines Adams,Defensive End,77.0,258.0,6.0,1.0,97.0,"{'city': None, 'state': None, 'country': None,..."
4,2092639.0,10547,213,Penn State,Big Ten,22,Arizona,2007,5,1,5,Levi Brown,Offensive Tackle,78.0,323.0,8.0,2.0,96.0,"{'city': None, 'state': None, 'country': None,..."


In [5]:
picks["pos_group"] = picks["position"].map(POSITION_MAP)   # rename "position" if the column is called something else
picks["pos_group"].value_counts(dropna=False)

pos_group
NaN    3821
Name: count, dtype: int64

In [6]:
print(stats["season"].agg(["min", "max"]), picks["year"].agg(["min", "max"]))

min    2007
max    2021
Name: season, dtype: int64 min    2007
max    2021
Name: year, dtype: int64


Everything looks to be accounted for, so now I'll start working with the data.

In [7]:
import numpy as np

### Cleaning Up the Picks

In [8]:
# --- Clean picks (reloads from disk, safe to rerun) ---
picks = pd.concat([pd.read_parquet(f) for f in sorted(RAW_COLLEGE.glob("draft_picks_*.parquet"))], ignore_index=True)

FULL = {"Quarterback":"QB","Running Back":"RB","Fullback":"FB","Wide Receiver":"WR","Tight End":"TE",
        "Offensive Tackle":"OT","Guard":"G","Offensive Guard":"G","Center":"C","Offensive Line":"OT",
        "Defensive End":"DE","Outside Linebacker":"OLB","Defensive Tackle":"DT","Nose Tackle":"NT",
        "Linebacker":"LB","Inside Linebacker":"ILB","Cornerback":"CB","Safety":"S","Free Safety":"FS",
        "Strong Safety":"SS","Kicker":"K","Place Kicker":"K","Punter":"P","Long Snapper":"LS",
        "Kick Returner":"KR","Defensive Back":"DB"}

picks = picks[picks["year"].between(CLASS_MIN, CLASS_MAX)].copy()
picks["pos_abbr"] = picks["position"].replace(FULL)
picks["pos_group"] = picks["pos_abbr"].map(POSITION_MAP)
picks["pid"] = picks["collegeAthleteId"].astype("Int64").astype(str)

# Defensive Backs → CB or S from college position (default CB)
col_pos = stats.assign(pid=stats["playerId"].astype(str)).groupby("pid")["position"].agg(lambda s: s.mode().iat[0])
db = picks["pos_abbr"] == "DB"
picks.loc[db, "pos_group"] = picks.loc[db, "pid"].map(col_pos).map(POSITION_MAP).fillna("CB")
print(f"{db.sum()} Defensive Backs →", picks.loc[db, "pos_group"].value_counts().to_dict())

excluded = ["K", "P", "LS", "KR", "PR"]
print("Unmapped:", picks.loc[picks["pos_group"].isna() & ~picks["pos_abbr"].isin(excluded), "position"].unique())
picks = picks.dropna(subset=["pos_group"])

# School level
POWER = {"ACC","Big 12","Big Ten","SEC","Pac-12","Pac-10","Big East"}
G5 = {"American Athletic","Mountain West","Mid-American","Sun Belt","Conference USA","FBS Independents","Western Athletic"}
picks["level"] = np.select([picks["collegeConference"].isin(POWER), picks["collegeConference"].isin(G5)],
                           ["Power", "G5/Ind"], "FCS/Other")
print(picks["pos_group"].value_counts())


3 Defensive Backs → {'CB': 3}
Unmapped: <ArrowStringArray>
['Fullback']
Length: 1, dtype: str
pos_group
EDGE    514
CB      384
WR      384
IOL     259
IDL     254
S       241
RB      241
OT      240
TE      164
LB      156
QB      141
Name: count, dtype: int64


Two things to note:
1. All of the Outside Linebackers (`OLB`) were mapped to Edge Rushers (`EDGE`), not Linebackers (`LB`). This is why there are so many more Edge Rushers (547) than Linebackers (177). This is according to `POSITION_MAP` in `src/utils`.
2. There were 3 players listed as Defensive Backs. My code tried to match them to either Safety (`S`) or Corner (`CB`), with Corner as the default option if no information was available. I'll confirm by hand rather than assuming.


In [9]:
db_rows = picks[picks["pos_abbr"] == "DB"][["name", "year", "collegeTeam", "pid", "weight"]]
db_rows["college_pos"] = db_rows["pid"].map(col_pos)
db_rows

,name,year,collegeTeam,pid,weight,college_pos
1233,Mikail Baker,2011,Baylor,173601,NaN,DB
1236,Malcolm Williams,2011,TCU,-1033734,NaN,NaN
1252,Jimmy Wilson,2011,Montana,2160152,NaN,NaN


After checking online, Mikail Baker and Malcolm Williams played CB, while Jimmy Wilson played Safety.

In [10]:
DB_OVERRIDES = {
    "Mikail Baker": "CB",
    "Malcolm Williams": "CB",
    "Jimmy Wilson": "S",
}
for name, grp in DB_OVERRIDES.items():
    mask = (picks["name"] == name) & (picks["pos_abbr"] == "DB")
    assert mask.sum() == 1, f"{name}: matched {mask.sum()} rows, check spelling"
    picks.loc[mask, "pos_group"] = grp

In [11]:
picks.loc[picks["pos_abbr"] == "DB", ["name", "year", "collegeTeam", "pos_group"]]

,name,year,collegeTeam,pos_group
1233,Mikail Baker,2011,Baylor,CB
1236,Malcolm Williams,2011,TCU,CB
1252,Jimmy Wilson,2011,Montana,S


Fixed! Next is...
### Cleaning up the Player Stats

In [12]:
# --- Clean stats ---
stats["pid"] = stats["playerId"].astype(str)
stats["value"] = pd.to_numeric(stats["stat"], errors="coerce")
stats["stat_name"] = stats["category"] + "." + stats["statType"]

d = stats.merge(picks[["pid", "year", "pos_group", "level"]], on="pid", how="inner")
d = d[d["season"] < d["year"]]                  # pre-draft seasons only
final = d[d["season"] == d["year"] - 1]

print("Non-numeric stats:", stats["value"].isna().sum())
print("Drafted players matched to any stats:", d["pid"].nunique(), "of", len(picks))
print("Matched in final season:", final["pid"].nunique())

Non-numeric stats: 0
Drafted players matched to any stats: 2093 of 2978
Matched in final season: 1904


There are no non-numeric stats, which makes my life easier, and a surprisingly high amount of picks matched to their college stats, given that most linemen don't record stats.

In [13]:
ol = d[d["pos_group"].isin(["OT", "IOL"])]
print(ol.groupby("stat_name")["value"].agg(rows="size", nonzero=lambda v: (v > 0).sum(), total="sum")
        .sort_values("nonzero", ascending=False))

                     rows  nonzero  total
stat_name                                
defensive.TOT         125      123  149.0
defensive.SOLO        125      104  118.0
receiving.REC          29       24   45.0
rushing.LONG           31       21  132.0
rushing.YDS            31       21  128.0
rushing.CAR            31       20   21.0
receiving.YDS          29       19  290.0
receiving.LONG         29       19  121.0
rushing.YPC            31       15   75.5
receiving.YPR          29       14   49.2
rushing.TD             31       14   15.0
fumbles.FUM             9        8    8.0
receiving.TD           29        4    4.0
kickReturns.LONG        3        3   26.0
kickReturns.NO          3        3    3.0
kickReturns.YDS         3        3   26.0
kickReturns.AVG         3        3   26.0
fumbles.LOST            9        2    2.0
defensive.TFL         125        2    2.0
passing.ATT             2        2    3.0
defensive.QB HUR      125        2    2.0
defensive.PD          125        1

Some linemen have defensive stats, likely from a tackle after an interception. Overall, this is not common and doesn't measure OL quality.

In [14]:
dn = d[d["value"] > 0]                                   # drop zero-value placeholder rows
picks["has_any"]   = picks["pid"].isin(dn["pid"])
picks["has_final"] = picks["pid"].isin(dn.loc[dn["season"] == dn["year"] - 1, "pid"])

coverage = (picks.groupby("pos_group")
            .agg(drafted=("pid", "size"), pct_any=("has_any", "mean"), pct_final=("has_final", "mean"))
            .reindex(POSITION_GROUPS).round(3))
coverage

,drafted,pct_any,pct_final
pos_group,,,
QB,141,0.929,0.922
RB,241,0.917,0.909
WR,384,0.906,0.891
TE,164,0.866,0.860
OT,240,0.258,0.142
IOL,259,0.274,0.131
EDGE,514,0.630,0.551
IDL,254,0.520,0.441
LB,156,0.917,0.833


If a player has any recorded college stats, they will be represented under `pct_any`. Meanwhile, `pct_final` is for players who have recorded college stats specifically in their last season before getting drafted. `pct_final` is slightly lower than `pct_any` because of injuries.

In [15]:
RELEVANT = {
    "QB":   ["passing.ATT", "passing.YDS", "passing.TD", "passing.INT", "rushing.YDS"],
    "RB":   ["rushing.CAR", "rushing.YDS", "rushing.TD", "receiving.REC", "receiving.YDS"],
    "WR":   ["receiving.REC", "receiving.YDS", "receiving.TD"],
    "TE":   ["receiving.REC", "receiving.YDS", "receiving.TD"],
    "OT":   [],
    "IOL":  [],
    "EDGE": ["defensive.SACKS", "defensive.TFL", "defensive.QB HUR", "defensive.TOT"],
    "IDL":  ["defensive.SACKS", "defensive.TFL", "defensive.QB HUR", "defensive.TOT"],
    "LB":   ["defensive.TOT", "defensive.SOLO", "defensive.TFL", "defensive.SACKS", "defensive.PD", "interceptions.INT"],
    "CB":   ["defensive.PD", "interceptions.INT", "defensive.TOT", "defensive.SOLO"],
    "S":    ["defensive.PD", "interceptions.INT", "defensive.TOT", "defensive.SOLO"],
}

Only including relevant stats for each position*

In [16]:
# Sanity check: every listed stat name actually exists in the data
all_names = set(d["stat_name"])
print("Missing stat names:", {s for v in RELEVANT.values() for s in v} - all_names)

Missing stat names: set()


Nothing missing!

In [17]:
# keeping only stats that are relevant to the players' positions
rel_pairs = pd.DataFrame([(g, s) for g, v in RELEVANT.items() for s in v], columns=["pos_group", "stat_name"])
dr = dn.merge(rel_pairs, on=["pos_group", "stat_name"])

In [18]:
picks["has_rel_any"]   = picks["pid"].isin(dr["pid"])
picks["has_rel_final"] = picks["pid"].isin(dr.loc[dr["season"] == dr["year"] - 1, "pid"])

In [19]:
coverage = (picks.groupby("pos_group")
            .agg(drafted=("pid", "size"),
                 pct_any=("has_any", "mean"),           # naive, from previous cell
                 rel_any=("has_rel_any", "mean"),
                 rel_final=("has_rel_final", "mean"))
            .reindex(POSITION_GROUPS).round(3))
coverage

,drafted,pct_any,rel_any,rel_final
pos_group,,,,
QB,141,0.929,0.929,0.922
RB,241,0.917,0.917,0.909
WR,384,0.906,0.906,0.888
TE,164,0.866,0.866,0.860
OT,240,0.258,0.000,0.000
IOL,259,0.274,0.000,0.000
EDGE,514,0.630,0.352,0.346
IDL,254,0.520,0.394,0.378
LB,156,0.917,0.917,0.821


There is a drop to 0 for the `OT` and `IOT` stats, which makes sense. However, there is also a noticeable drop for `LB`, of almost 40 percentage points. I'll look into this further.

In [20]:
# defenders who have nonzero stats but none of the relevant ones: what do they have?
gap = picks[picks["has_any"] & ~picks["has_rel_any"] & picks["pos_group"].isin(["LB", "EDGE", "IDL"])]
print(len(gap), "players")
print(dn[dn["pid"].isin(gap["pid"])]["stat_name"].value_counts().head(15))

175 players
stat_name
interceptions.INT    229
interceptions.AVG    184
interceptions.YDS    184
interceptions.TD      68
puntReturns.NO        33
puntReturns.AVG       30
puntReturns.LONG      30
puntReturns.YDS       30
receiving.LONG        20
receiving.REC         20
receiving.YDS         20
receiving.YPR         20
kickReturns.NO        20
rushing.CAR           19
rushing.LONG          18
Name: count, dtype: int64


The biggest stat to include is interceptions, as they are significant on defense, for `LB` and even `EDGE`. The other special teams and offensive stats, meanwhile, do not add much.

In [21]:
# spot-check a few: name/position in picks vs. what CFBD stats say
check = gap[["pid", "name", "pos_group", "collegeTeam", "year"]].head(10).merge(
    stats[["pid", "player", "position", "team"]].drop_duplicates("pid"), on="pid", how="left")
check

,pid,name,pos_group,collegeTeam,year,player,position,team
0,175300,Ndamukong Suh,IDL,Nebraska,2010,Ndamukong Suh,DT,Nebraska
1,191442,Brandon Graham,EDGE,Michigan,2010,Brandon Graham,DE,Michigan
2,482895,Jason Pierre-Paul,EDGE,South Florida,2010,Jason Pierre-Paul,DE,South Florida
3,191607,Sean Weatherspoon,EDGE,Missouri,2010,Sean Weatherspoon,LB,Missouri
4,238495,Linval Joseph,IDL,East Carolina,2010,Linval Joseph,DL,East Carolina
5,189938,Daryl Washington,EDGE,TCU,2010,Daryl Washington,LB,TCU
6,188306,Thaddeus Gibson,EDGE,Ohio State,2010,Thaddeus Gibson,LB,Ohio State
7,188288,A.J. Edds,EDGE,Iowa,2010,A.J. Edds,LB,Iowa
8,174556,Keenan Clayton,EDGE,Oklahoma,2010,Keenan Clayton,LB,Oklahoma
9,190688,Eric Norwood,EDGE,South Carolina,2010,Eric Norwood,LB,South Carolina


This is interesting becuase there are notable players with productive seasons on this list, yet the CFBD dataset doesn't show any recorded statistics. These all seem to be in the early seasons, so I'll look into if that is the trend/issue.

In [22]:
# share of all defensive players (not just drafted) with any defensive.* stat, by season
def_pos = ["DL", "DT", "DE", "LB", "DB", "CB", "S"]
s = stats[stats["position"].isin(def_pos)]
has_def = s[s["category"] == "defensive"].groupby("season")["pid"].nunique()
all_def = s.groupby("season")["pid"].nunique()
(has_def / all_def).round(2)

season
2007     NaN
2008     NaN
2009     NaN
2010     NaN
2011     NaN
2012     NaN
2013     NaN
2014     NaN
2015     NaN
2016    0.99
2017    0.99
2018    0.97
2019    0.99
2020    0.98
2021    0.96
Name: pid, dtype: float64

It looks like there was no defensive category up until 2016. This explains why players in 2010 like Suh had interceptions but no sacks or tackles. But before accepting this, I'll confirm.

In [23]:
for yr in [2012, 2015, 2016]:
    print(yr, sorted(stats.loc[stats["season"] == yr, "category"].unique()))

2012 ['interceptions', 'kickReturns', 'kicking', 'passing', 'puntReturns', 'punting', 'receiving', 'rushing']
2015 ['interceptions', 'kickReturns', 'kicking', 'passing', 'puntReturns', 'punting', 'receiving', 'rushing']
2016 ['defensive', 'fumbles', 'interceptions', 'kickReturns', 'kicking', 'passing', 'puntReturns', 'punting', 'receiving', 'rushing']


Yup, it looks like there are no fumbles or defensive stats before 2016. One last thing I can do is call the API to see if the pull script missed it.

In [24]:
import os, requests
from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

r = requests.get("https://api.collegefootballdata.com/stats/player/season",
                 params={"year": 2013, "category": "defensive"},
                 headers={"Authorization": f"Bearer {os.environ['CFBD_API_KEY']}"})
print(r.status_code, len(r.json()))
print(r.json()[:3])

200 0
[]


Yup, it came up empty.

In [25]:
r = requests.get("https://api.collegefootballdata.com/games/players",
                 params={"year": 2013, "week": 5},
                 headers={"Authorization": f"Bearer {os.environ['CFBD_API_KEY']}"})
games = r.json()
cats = {c["name"] for g in games for t in g["teams"] for c in t["categories"]}
print(len(games), sorted(cats))

49 ['interceptions', 'kickReturns', 'kicking', 'passing', 'puntReturns', 'punting', 'receiving', 'rushing']


And after checking the box score, still no defensive or fumble statistics.

This is very important. CFBD has no defensive stats (tackles, sacks, TFL, hurries) before the 2016 season, even after directly confirming with the API. The only defensive statistic available pre-2016 is interceptions. That means most defenders in the 2010–2016 classes have no college defensive production that we can measure. 

This is not an issue with the offense, though (offense covers all of 2010–2022).

Options: 
1. Train defensive Layer 2 on the 2017–22 classes only, 
2. Use combine and pick slot only for earlier defenders, 
3. Add another source for 2008–15 (which means a scraper and name matching). 

Which do we want?

In [26]:
# tag each drafted defender with an era and whether they have any defensive.* stat
DEF_GROUPS = ["EDGE", "IDL", "LB", "CB", "S"]
has_def = dn.loc[dn["category"] == "defensive", "pid"]
era = (picks[picks["pos_group"].isin(DEF_GROUPS)]
       .assign(era=lambda x: np.where(x["year"] <= 2016, "2010-16", "2017-22"),
               has_def_any=lambda x: x["pid"].isin(has_def)))

Splits drafted defenders into the 2010–16 classes (final season before CFBD added defensive stats) and the 2017–22 classes, and flags who has any tackle/sack/TFL-type stat.

In [27]:
era_table = (era.groupby(["pos_group", "era"])
                .agg(drafted=("pid", "size"),
                     rel_final=("has_rel_final", "mean"),
                     def_any=("has_def_any", "mean"))
                .unstack("era")
                .reindex(DEF_GROUPS).round(3))
era_table

drafted         rel_final         def_any        
era       2010-16 2017-22   2010-16 2017-22 2010-16 2017-22
pos_group                                                  
EDGE          317     197     0.000   0.904     0.0   0.919
IDL           148     106     0.000   0.906     0.0   0.943
LB             79      77     0.696   0.948     0.0   0.974
CB            224     159     0.643   0.906     0.0   0.943
S             134     108     0.612   0.917     0.0   0.963

Era table: final-season relevant coverage (`rel_final`) and share with any `defensive.*` stat (`def_any`), before vs. after 2016.

`def_any` is 0 for every position in 2010–16, which confirms the gap. EDGE and IDL drop to 0 coverage there. LB, CB, and S keep 61–70% only because of interceptions, so for those early classes "covered" means "we know their INT count and nothing else." From 2017 on, every defensive group is about 89–96% covered.

In [28]:
# share of players in each position group with a nonzero value for each stat
# 2017-22 classes only, so the pre-2016 defensive gap doesn't hide which stats exist
p = picks[(picks["year"] >= 2017) & picks["has_any"]]
base = p.groupby("pos_group")["pid"].nunique()
avail = (dn[dn["pid"].isin(p["pid"])]
         .groupby(["pos_group", "stat_name"])["pid"].nunique()
         .div(base, level="pos_group").rename("share").reset_index())

Builds `avail`: for each position group and stat, the share of drafted players (2017–22 classes, with any stats) who have a nonzero value in a pre-draft season.

In [29]:
# how available is each stat we picked in RELEVANT? flag anything under 10%
rel_check = rel_pairs.merge(avail, how="left").fillna({"share": 0})
rel_check["rare"] = rel_check["share"] < 0.10
rel_check.round(3)

,pos_group,stat_name,share,rare
0,QB,passing.ATT,1.000,False
1,QB,passing.YDS,1.000,False
2,QB,passing.TD,1.000,False
3,QB,passing.INT,1.000,False
4,QB,rushing.YDS,0.889,False
5,RB,rushing.CAR,1.000,False
6,RB,rushing.YDS,1.000,False
7,RB,rushing.TD,1.000,False
8,RB,receiving.REC,1.000,False
9,RB,receiving.YDS,1.000,False


Checks every stat in `RELEVANT` against `avail`.

None are rare: every chosen stat is present for at least 90% of the position, except LB interceptions (78%). That makes sense, since many linebackers never record one.

In [30]:
# stats NOT in RELEVANT that are common for the position: top 5 per group
others = (avail.merge(rel_pairs, how="left", indicator=True)
               .query("_merge == 'left_only'").drop(columns="_merge")
               .sort_values(["pos_group", "share"], ascending=[True, False])
               .groupby("pos_group").head(5))
others.round(3)

,pos_group,stat_name,share
5,CB,defensive.TFL,0.921
10,CB,interceptions.AVG,0.795
13,CB,interceptions.YDS,0.795
9,CB,fumbles.REC,0.430
4,CB,defensive.TD,0.384
42,EDGE,defensive.SOLO,1.000
39,EDGE,defensive.PD,0.884
48,EDGE,fumbles.REC,0.713
50,EDGE,interceptions.INT,0.403
49,EDGE,interceptions.AVG,0.326


Shows the five most common stats each position has that aren't in `RELEVANT`, to catch anything I left out.

A few look worth adding when I build the candidate features: QB `rushing.CAR`, LB `defensive.QB HUR`, and CB/S `defensive.TFL`. Rate stats like `passing.PCT`, `passing.YPA`, and `rushing.YPC` can be computed from the counts, so they don't need to be pulled separately. `defensive.TOT` showing up for offensive players and OL is just post-turnover tackles (same as the OL check earlier), so I'm ignoring it.

In [31]:
# missing = no nonzero relevant stat in any pre-draft season (OT/IOL left out: 100% by definition)
SKILL = [g for g in POSITION_GROUPS if RELEVANT[g]]
m = picks[picks["pos_group"].isin(SKILL)].assign(missing=lambda x: ~x["has_rel_any"])

Builds `m`: one row per drafted player (OT/IOL excluded), with `missing = True` if they have no nonzero relevant stat in any pre-draft season.

In [32]:
miss_year = (m.pivot_table(index="pos_group", columns="year", values="missing", aggfunc="mean")
               .reindex(SKILL).round(2))
miss_year

year,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021
pos_group,,,,,,,,,,,,
QB,0.14,0.17,0.09,0.09,0.00,0.00,0.07,0.10,0.08,0.00,0.00,0.10
RB,0.23,0.20,0.16,0.09,0.00,0.16,0.05,0.08,0.05,0.00,0.00,0.00
WR,0.19,0.18,0.12,0.14,0.12,0.14,0.12,0.09,0.03,0.00,0.00,0.03
TE,0.30,0.25,0.09,0.00,0.00,0.16,0.44,0.21,0.00,0.00,0.08,0.09
EDGE,1.00,1.00,1.00,1.00,1.00,1.00,1.00,0.21,0.09,0.05,0.03,0.03
IDL,1.00,1.00,1.00,1.00,1.00,1.00,1.00,0.09,0.19,0.00,0.00,0.00
LB,0.15,0.17,0.11,0.09,0.12,0.15,0.15,0.08,0.00,0.07,0.00,0.00
CB,0.32,0.34,0.27,0.17,0.33,0.07,0.31,0.09,0.06,0.10,0.00,0.00
S,0.21,0.39,0.40,0.22,0.24,0.25,0.19,0.08,0.06,0.00,0.04,0.00


Missingness by draft class (share of each position with no relevant stats).

EDGE and IDL are 100% missing for 2010–16 (the defensive gap), then drop to about 0–20%. Offense is missing 10–20% in the earliest classes and close to 0 after 2018. The 2010–11 classes are worse partly because their pre-draft history starts in 2008, when the raw files begin. TE swings a lot year to year because each class is small.

In [33]:
miss_level = (m.pivot_table(index="pos_group", columns="level", values="missing", aggfunc="mean")
                .reindex(SKILL).round(2))
n_level = m.pivot_table(index="pos_group", columns="level", values="pid", aggfunc="size").reindex(SKILL)
display(miss_level, n_level)

level,FCS/Other,G5/Ind,Power
pos_group,,,
QB,0.67,0.03,0.03
RB,0.50,0.04,0.07
WR,0.79,0.06,0.03
TE,0.83,0.14,0.06
EDGE,0.87,0.62,0.63
IDL,0.91,0.65,0.57
LB,1.00,0.00,0.08
CB,0.81,0.11,0.10
S,0.85,0.15,0.09


level,FCS/Other,G5/Ind,Power
pos_group,,,
QB,9,30,102
RB,12,47,182
WR,28,84,272
TE,12,37,115
EDGE,46,91,377
IDL,22,34,198
LB,3,25,128
CB,42,80,261
S,20,39,183


Missingness by school level, with the player counts underneath.

FCS/Other is the clear problem: 50–88% missing at every position, likely because CFBD's season stats mostly cover FBS teams. Power and G5 are close to each other on offense (3–12%). For defense, they're both around 55–60% for EDGE/IDL, which is the pre-2016 gap again, not school level. FCS counts are small (4 LBs, 10 QBs), so those rates are noisy.

In [34]:
# picks with no stats at all: unlinked CFBD id (negative) vs. stats filed under a different id
no_stats = picks[~picks["has_any"]]
unlinked = no_stats["pid"].str.startswith("-")
names = stats[["player", "team", "season"]].drop_duplicates()
by_name = (no_stats[~unlinked]
           .merge(names, left_on=["name", "collegeTeam"], right_on=["player", "team"])
           .query("season < year"))
print(f"{len(no_stats)} picks with no stats: {unlinked.sum()} have an unlinked (negative) id, "
      f"{by_name['pid'].nunique()} have stats under a different id (exact name + school)")
by_name.drop_duplicates("pid")["year"].value_counts().sort_index()

885 picks with no stats: 464 have an unlinked (negative) id, 54 have stats under a different id (exact name + school)


year
2010    27
2011     8
2012     2
2013     4
2015     3
2016     5
2017     2
2018     3
Name: count, dtype: int64

Breaks down the 920 picks with no stats at all.

465 have a negative `collegeAthleteId`, which is how CFBD marks a pick that isn't linked to a college player record, so the ID join can never find them. Another 44 do have stats, but filed under a different `playerId` (for example, Rob Gronkowski and Jermaine Gresham), and most of those are in the 2010 class. So the direct ID join is clean when it hits, but it misses some players. I'm leaving these unmatched here and flagging it for the master-table matching instead.

In [35]:
# one headline stat per position, from the final pre-draft season
# uses `final` (keeps real zeros: a row exists only if the player has that category that season)
KEY_STAT = {"QB": "passing.YDS", "RB": "rushing.YDS", "WR": "receiving.YDS", "TE": "receiving.YDS",
            "EDGE": "defensive.SACKS", "IDL": "defensive.TFL", "LB": "defensive.TOT",
            "CB": "defensive.PD", "S": "defensive.TOT"}
key_pairs = pd.DataFrame(KEY_STAT.items(), columns=["pos_group", "stat_name"])
fk = final.merge(key_pairs, on=["pos_group", "stat_name"])

Builds `fk`: one row per player with the final-season value of their position's headline stat (`KEY_STAT`).

I used `final` instead of `dn` on purpose. A 0 here is a real zero (for example, an EDGE with tackles but no sacks), not a placeholder, because the row only exists if the player has stats in that category.

In [36]:
fk.groupby("pos_group")["value"].describe().reindex(list(KEY_STAT)).round(1)

,count,mean,std,min,25%,50%,75%,max
pos_group,,,,,,,,
QB,130.0,3317.7,929.2,156.0,2843.8,3351.5,3900.8,5671.0
RB,219.0,1102.0,502.7,16.0,728.0,1114.0,1416.5,2587.0
WR,341.0,914.9,376.8,-3.0,670.0,904.0,1184.0,1856.0
TE,141.0,457.0,235.7,19.0,281.0,457.0,598.0,1352.0
EDGE,178.0,6.1,3.6,0.0,3.5,5.5,8.5,17.0
IDL,96.0,8.4,4.3,1.0,5.4,8.0,11.5,19.5
LB,73.0,83.5,32.4,14.0,62.0,85.0,102.0,181.0
CB,144.0,7.5,4.4,0.0,4.0,7.5,11.0,19.0
S,99.0,63.3,22.0,6.0,50.0,65.0,76.0,118.0


Summary of each headline stat (count, mean, spread, min/max).

The ranges look like real football: the median drafted QB threw about 3,300 yards, the median EDGE had 5.5 sacks. The small negative minimums for RB/WR are real (a few carries for a loss), not errors.

In [37]:
# the two highest values per position: are the outliers real seasons or data errors?
(fk.sort_values("value", ascending=False).groupby("pos_group").head(2)
   [["pos_group", "player", "season", "team", "stat_name", "value"]].sort_values("pos_group"))

,pos_group,player,season,team,stat_name,value
617,CB,Ahkello Witherspoon,2016,Colorado,defensive.PD,19.0
781,CB,Joshua Jackson,2017,Iowa,defensive.PD,18.0
929,EDGE,Josh Allen,2018,Kentucky,defensive.SACKS,17.0
1200,EDGE,Chase Young,2019,Ohio State,defensive.SACKS,16.5
1020,IDL,Quinnen Williams,2018,Alabama,defensive.TFL,19.5
1223,IDL,James Lynch,2019,Baylor,defensive.TFL,19.5
1122,LB,Evan Weaver,2019,California,defensive.TOT,181.0
912,LB,Ben Burr-Kirven,2018,Washington,defensive.TOT,165.0
1067,QB,Joe Burrow,2019,LSU,passing.YDS,5671.0
275,QB,Derek Carr,2013,Fresno State,passing.YDS,5082.0


Lists the top two seasons per position to check the outliers.

They're all real, well-known seasons (Melvin Gordon's 2,587 rushing yards, Joe Burrow's 2019, Chase Young's 16.5 sacks), so there's nothing to clean.

In [38]:
lvl_median = fk.pivot_table(index="pos_group", columns="level", values="value", aggfunc="median").reindex(list(KEY_STAT))
lvl_n = fk.pivot_table(index="pos_group", columns="level", values="value", aggfunc="size").reindex(list(KEY_STAT))
display(lvl_median, lvl_n)

level,FCS/Other,G5/Ind,Power
pos_group,,,
QB,483.5,3581.0,3186.0
RB,102.5,1133.0,1140.0
WR,152.0,1115.5,864.5
TE,136.0,509.5,425.5
EDGE,0.0,7.0,5.5
IDL,1.0,10.5,7.5
LB,NaN,87.0,84.5
CB,0.0,8.0,8.0
S,7.0,72.5,65.0


level,FCS/Other,G5/Ind,Power
pos_group,,,
QB,2.0,29.0,99.0
RB,6.0,44.0,169.0
WR,5.0,78.0,258.0
TE,1.0,32.0,108.0
EDGE,5.0,35.0,138.0
IDL,2.0,12.0,82.0
LB,NaN,15.0,58.0
CB,7.0,33.0,104.0
S,2.0,12.0,85.0


Median headline stat by school level, with player counts underneath.

Drafted G5 players have higher raw numbers than Power players (WR 1,117 vs. 866 yards, QB 3,555 vs. 3,187). That's likely because a G5 player has to dominate weaker opponents to get drafted, so raw counts aren't comparable across levels. FCS medians are tiny (QB 224 yards, EDGE 0 sacks), which suggests CFBD only has some of their games. With 1–7 FCS players per position, I'd treat FCS college stats as missing instead of trusting them.

In [39]:
season_median = fk.pivot_table(index="pos_group", columns="season", values="value", aggfunc="median").reindex(list(KEY_STAT))
season_median

season,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020
pos_group,,,,,,,,,,,,
QB,2847.5,2995.5,3630.5,3437.5,3554.5,3855.0,3488.0,2946.0,3688.5,3555.00,2951.0,2830.0
RB,1278.5,1157.0,1265.5,1081.0,1139.0,1092.5,1078.0,1176.5,1227.0,990.00,1117.0,858.0
WR,927.0,899.5,956.0,874.0,1060.0,863.0,926.0,968.0,879.0,931.50,902.5,625.0
TE,493.0,358.0,499.5,466.5,478.5,293.0,438.0,651.0,477.0,472.00,504.0,294.0
EDGE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.5,4.5,6.50,5.5,5.0
IDL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.5,7.5,9.75,8.5,4.5
LB,NaN,NaN,NaN,NaN,NaN,NaN,NaN,96.0,85.5,82.00,84.5,81.0
CB,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.5,9.0,9.00,8.0,4.0
S,NaN,NaN,NaN,NaN,NaN,NaN,NaN,71.0,76.5,65.00,68.5,42.0


Median headline stat by final season, to look for drift over time.

Most positions are fairly stable year to year, with noise from small groups (about 9–40 players per cell). The clear exception is 2020, the COVID-shortened season: WR (625), S (42), and CB (4.0) all drop sharply, because players got fewer games. Since CFBD has no games-played column, we can't convert to per-game rates. So I'd recommend standardizing each stat within season and position (for example, a percentile or z-score among that year's drafted players at the position), and adding school level as its own feature.

## Candidate features

In [40]:
# NCAA passer efficiency from final-season counts; under 100 attempts counts as missing (too noisy)
qb = (final[(final["pos_group"] == "QB") & (final["category"] == "passing")]
      .pivot_table(index=["pid", "player", "team", "season", "level"], columns="statType", values="value")
      .reset_index())
qb["pass_eff"] = (8.4*qb["YDS"] + 330*qb["TD"] + 100*qb["COMPLETIONS"] - 200*qb["INT"]) / qb["ATT"]
qb.loc[qb["ATT"] < 100, "pass_eff"] = np.nan

Computes each QB's final-season passer efficiency rating, the college formula, from attempts, completions, yards, TDs, and INTs. QBs with fewer than 100 attempts get `NaN`.

In [41]:
display(qb["pass_eff"].describe().round(1),
        qb.groupby("level")["pass_eff"].agg(["median", "count"]).round(1))

count    127.0
mean     151.8
std       19.4
min      105.9
25%      138.0
50%      149.8
75%      163.4
max      206.9
Name: pass_eff, dtype: float64

,median,count
level,,
FCS/Other,NaN,0
G5/Ind,150.7,29
Power,147.9,98


Distribution of passer efficiency, overall and by school level.

135 of 139 QBs have 100+ attempts. The median is about 150 and the top is Tua Tagovailoa's 2019 (206.9). Joe Burrow's 2019 comes out at 202.0, which matches his official rating, so the formula is right. Unlike raw passing yards, efficiency is nearly the same at G5 and Power (152 vs. 150), so a per-attempt stat is less tilted by level than a total.

In [42]:
# final-season relevant coverage by era, for every position group
cov = (picks.assign(era=np.where(picks["year"] <= 2016, "2010-16", "2017-22"))
            .pivot_table(index="pos_group", columns="era", values="has_rel_final", aggfunc="mean")
            .reindex(POSITION_GROUPS).round(2))

Builds `cov`: the share of each position with a relevant final-season stat, split by era, to sit next to the features below.

In [43]:
# candidate college features per position group (final season, plus career totals over pre-draft seasons)
# every group also gets: school level, combine + size, and pick slot (from other members' tables)
CANDIDATES = {
    "QB":   ("passing ATT/YDS/TD/INT, pass_eff (100+ att), rushing CAR/YDS", "career pass YDS/TD, seasons with stats"),
    "RB":   ("rushing CAR/YDS/TD, receiving REC/YDS, YPC",                    "career rush YDS, career REC"),
    "WR":   ("receiving REC/YDS/TD, YPR",                                      "career rec YDS/TD"),
    "TE":   ("receiving REC/YDS/TD, YPR",                                      "career rec YDS/TD"),
    "OT":   ("none",                                                           "none"),
    "IOL":  ("none",                                                           "none"),
    "EDGE": ("SACKS, TFL, QB HUR, TOT",                                        "career SACKS/TFL (2017+ classes)"),
    "IDL":  ("TFL, SACKS, QB HUR, TOT",                                        "career TFL/SACKS (2017+ classes)"),
    "LB":   ("TOT, SOLO, TFL, SACKS, QB HUR, PD, INT",                        "career TOT/TFL (2017+), career INT (all)"),
    "CB":   ("PD, INT, TOT, SOLO, TFL",                                        "career PD (2017+), career INT (all)"),
    "S":    ("PD, INT, TOT, SOLO, TFL",                                        "career TOT/PD (2017+), career INT (all)"),
}
CONCERNS = {
    "QB":   "small group (~150); efficiency noisy under 100 att",
    "RB":   "counting stats favor durable players",
    "WR":   "counting stats favor durable players",
    "TE":   "small classes; blocking TEs have few catches",
    "OT":   "no college production; size, combine, pick slot only",
    "IOL":  "no college production; size, combine, pick slot only",
    "EDGE": "no defensive stats before 2016; OLB->EDGE mapping inflates group",
    "IDL":  "no defensive stats before 2016",
    "LB":   "pre-2016 classes covered by INTs only",
    "CB":   "pre-2016 classes covered by INTs only",
    "S":    "pre-2016 classes covered by INTs only",
}
features = (pd.DataFrame.from_dict(CANDIDATES, orient="index", columns=["final_season", "career"])
              .assign(concerns=pd.Series(CONCERNS))
              .join(cov.add_prefix("rel_final_")))
features

,final_season,career,concerns,rel_final_2010-16,rel_final_2017-22
QB,"passing ATT/YDS/TD/INT, pass_eff (100+ att), r...","career pass YDS/TD, seasons with stats",small group (~150); efficiency noisy under 100...,0.92,0.93
RB,"rushing CAR/YDS/TD, receiving REC/YDS, YPC","career rush YDS, career REC",counting stats favor durable players,0.87,0.96
WR,"receiving REC/YDS/TD, YPR",career rec YDS/TD,counting stats favor durable players,0.84,0.95
TE,"receiving REC/YDS/TD, YPR",career rec YDS/TD,small classes; blocking TEs have few catches,0.82,0.91
OT,none,none,"no college production; size, combine, pick slo...",0.00,0.00
IOL,none,none,"no college production; size, combine, pick slo...",0.00,0.00
EDGE,"SACKS, TFL, QB HUR, TOT",career SACKS/TFL (2017+ classes),no defensive stats before 2016; OLB->EDGE mapp...,0.00,0.90
IDL,"TFL, SACKS, QB HUR, TOT",career TFL/SACKS (2017+ classes),no defensive stats before 2016,0.00,0.91
LB,"TOT, SOLO, TFL, SACKS, QB HUR, PD, INT","career TOT/TFL (2017+), career INT (all)",pre-2016 classes covered by INTs only,0.70,0.95
CB,"PD, INT, TOT, SOLO, TFL","career PD (2017+), career INT (all)",pre-2016 classes covered by INTs only,0.64,0.91


The candidate feature table: final-season and career stats per position, with coverage by era and the main concern for each group.

Every position also gets school level, combine and size measurements, and pick slot, all standardized within season and position as recommended in step 5. OT and IOL have no usable college production, so their models rely on those shared inputs alone. For EDGE and IDL, college features only exist for the 2017–22 classes, so that's the team decision from earlier. Career totals are short for the 2010–11 classes, since the raw data starts with the 2008 season. These are candidates, to be tested against the pick-slot baseline in Layer 2, not a final list.

## Pre-2016 defensive fill (ESPN play-by-play)

cfbfastR's `cfbd_*` functions wrap the same CFBD API, and ESPN's box scores have the same pre-2016 gap. But sportsdataverse (the project behind cfbfastR) publishes `espn_cfb_adv_defensive_players`, which counts sacks, pass breakups, INTs, and forced fumbles from ESPN play-by-play text, back to 2004. Testing whether it can fill the gap.

In [44]:
# per-game defensive events parsed from ESPN play-by-play, read straight from the sportsdataverse release
ESPN_DEF_URL = ("https://github.com/sportsdataverse/sportsdataverse-data/releases/download/"
                "espn_cfb_adv_defensive_players/adv_defensive_players_{}.parquet")
espn_raw = pd.concat([pd.read_parquet(ESPN_DEF_URL.format(y)) for y in range(2008, 2018)], ignore_index=True)
espn_raw = espn_raw[espn_raw["player_name"] != "Team"]       # team-credited plays, not a player
espn_raw.groupby("season")["sacks"].sum()

season
2008    2246.0
2009    2644.5
2010    2991.5
2011    3022.5
2012    3190.5
2013      17.5
2014    3548.0
2015    3505.0
2016    3550.0
2017    3570.0
Name: sacks, dtype: float64

Loads the 2008–2017 seasons and shows total sacks per season.

Every season has about 2,300–3,700 sacks except **2013, which has 18**. The 2013 play text must be missing the sacker's name. So 2013 sacks are unusable, and that's the final season for the 2014 class. Pass breakups and forced fumbles look normal in 2013.

In [45]:
# match key: last name + first initial (handles Chris/Christopher, and junk prefixes like "vandy andre hal")
def espn_key(names):
    tokens = (names.str.lower().str.replace(r"[^a-z ]", "", regex=True)
                   .str.replace(r"\b(jr|sr|ii|iii|iv)\b", "", regex=True).str.split())
    return tokens.str[-1] + "_" + tokens.str[-2].str[0]

ESPN_COLS = ["sacks", "pass_breakups", "interceptions", "forced_fumbles"]
espn_def = (espn_raw.assign(key=espn_key(espn_raw["player_name"])).dropna(subset=["key"])
                    .groupby(["season", "def_pos_team_id", "key"], as_index=False)[ESPN_COLS].sum())

Builds `espn_def`: one row per player-season, keyed by school id + last name + first initial.

ESPN names have no player id and are messy ("Lamin Barrow.", "Buff Khalil Mack"), and an exact-name match missed nicknames. Last name + first initial fixes most of that. The cost: two players at the same school with the same last name and first initial would be merged.

In [46]:
# join drafted defenders on school id (picks.collegeId is ESPN's team id) + key, pre-draft seasons only
def_picks = (picks[picks["pos_group"].isin(DEF_GROUPS)]
             .assign(key=lambda x: espn_key(x["name"]), collegeId=lambda x: x["collegeId"].astype(int)))
fill = def_picks.merge(espn_def, left_on=["collegeId", "key"], right_on=["def_pos_team_id", "key"])
fill = fill[fill["season"] < fill["year"]]
print("drafted defenders sharing school + key with another pick:", def_picks.duplicated(["collegeId", "key"], keep=False).sum())

drafted defenders sharing school + key with another pick: 18


Builds `fill`: each drafted defender's ESPN player-seasons before the draft.

20 drafted defenders share a school and key with another drafted defender, so a few rows could belong to the wrong player. That's small, but worth remembering if a number looks off.

In [47]:
# validation: where both sources exist (2016-17 seasons), do ESPN sacks agree with CFBD sacks?
cfbd_sacks = d.loc[(d["stat_name"] == "defensive.SACKS") & d["season"].isin([2016, 2017]), ["pid", "season", "value"]]
overlap = fill[fill["season"].isin([2016, 2017])].merge(cfbd_sacks, on=["pid", "season"])
diff = (overlap["sacks"] - overlap["value"]).abs()
print(f"{len(overlap)} player-seasons | exact: {(diff == 0).mean():.3f} | within 1: {(diff <= 1).mean():.3f} | "
      f"corr: {overlap[['sacks', 'value']].corr().iloc[0, 1]:.3f}")

709 player-seasons | exact: 0.772 | within 1: 0.968 | corr: 0.961


Compares ESPN's sacks with CFBD's for the same players in 2016–17, where CFBD has defense.

78% agree exactly and 97% are within one sack, with a correlation of 0.96, and the average difference is about 0. ESPN's counts are slightly noisy (likely shared sacks and plays credited differently), but they aren't biased.

In [48]:
# coverage gain for the 2010-16 classes: share with any ESPN row (any season, and final season)
gap = (def_picks[def_picks["year"] <= 2016]
       .assign(espn_any=lambda x: x["pid"].isin(fill["pid"]),
               espn_final=lambda x: x["pid"].isin(fill.loc[fill["season"] == fill["year"] - 1, "pid"])))
display(gap.groupby("pos_group")[["espn_any", "espn_final"]].mean().reindex(DEF_GROUPS).round(3),
        gap.groupby("level")[["espn_any"]].agg(["mean", "size"]).round(3),
        gap.groupby("year")[["espn_any", "espn_final"]].mean().round(3).T)

,espn_any,espn_final
pos_group,,
EDGE,0.956,0.899
IDL,0.912,0.845
LB,0.962,0.949
CB,0.960,0.884
S,0.948,0.896


espn_any     
              mean size
level                  
FCS/Other    0.570   93
G5/Ind       0.994  161
Power        0.992  648

year,2010,2011,2012,2013,2014,2015,2016
espn_any,0.964,0.946,0.960,0.898,0.907,0.992,0.976
espn_final,0.927,0.892,0.929,0.883,0.729,0.968,0.906


Coverage for the 2010–16 classes after the fill: by position, by school level, and by draft class.

Power and G5 defenders now match 99% of the time, and 91–96% at every position, up from 0% defensive coverage for EDGE/IDL. FCS/Other only matches 57%, likely because ESPN's play-by-play covers fewer FCS games. The 2014 class is the weak spot at 73% final-season coverage, and its final-season sacks are unusable because of the 2013 problem.

In [49]:
# spot-check well-known pre-2016 pass rushers
(fill[fill["name"].isin(["Ndamukong Suh", "Von Miller", "Jadeveon Clowney", "Khalil Mack", "J.J. Watt"])]
     .sort_values(["name", "season"])[["name", "season", "collegeTeam"] + ESPN_COLS])

,name,season,collegeTeam,sacks,pass_breakups,interceptions,forced_fumbles
256,J.J. Watt,2009,Wisconsin,6.0,1.0,0.0,0.0
257,J.J. Watt,2010,Wisconsin,6.0,3.0,1.0,3.0
1267,Jadeveon Clowney,2011,South Carolina,5.0,0.0,0.0,2.0
1268,Jadeveon Clowney,2012,South Carolina,14.0,0.0,0.0,2.0
1269,Jadeveon Clowney,2013,South Carolina,0.0,3.0,0.0,0.0
1270,Khalil Mack,2010,Buffalo,4.5,7.0,0.0,0.0
1271,Khalil Mack,2011,Buffalo,4.5,2.0,1.0,2.0
1272,Khalil Mack,2012,Buffalo,5.5,1.0,0.0,1.0
1273,Khalil Mack,2013,Buffalo,0.5,3.0,3.0,2.0
0,Ndamukong Suh,2008,Nebraska,7.5,3.0,2.0,0.0


Shows the season-by-season ESPN numbers for five well-known pass rushers.

The big seasons show up: Suh's 2009 (12.5 sacks), Von Miller's 2009 (16), and Clowney's 2012 (14). These are close to their well-known totals but not always exact, which fits the validation above. Clowney's and Mack's 2013 seasons show 0–0.5 sacks, which is the 2013 problem, not real production.